# Read k-space from HDF5 (fastMRI-style) and reconstruct images
This notebook loads k-space from an `.h5` file, supports single-coil and multi-coil layouts, performs iFFT reconstruction, and visualizes the image and k-space magnitude.

In [ ]:
import os, glob, json
import numpy as np
import h5py
import torch
import matplotlib.pyplot as plt
from fastmri import ifft2c, fft2c, rss_complex

# Optional: ISMRMRD support
import ismrmrd
from ismrmrd import xsd as ismrmrd_xsd
from pprint import pprint


In [ ]:
# Find candidate .h5 files in common dataset locations
data_path = "/home/paula/SDF_intro/MRI_new_dataset/data_samples/mri_new_data/h5_v3"

candidates = []
for root, _, _ in os.walk(data_path):
    candidates.extend(glob.glob(os.path.join(root, '*.h5')))
print(f'Found {len(candidates)} .h5 files')
# Pick the first by default; set `filename` manually if you prefer another
filename = candidates[0] if candidates else ''
filename

In [ ]:
def load_h5_kspace(path):
    """
    Load k-space from an HDF5 file.
    Supports:
      - fastMRI-like HDF5 ('kspace' dataset)
      - ISMRMRD HDF5 produced by siemens_to_ismrmrd (dataset + acquisitions)
    Returns:
      kspace: np.ndarray with shape (S, C, H, W, 2) and dtype float32
      info  : dict with basic metadata
    """
    if not path or not os.path.exists(path):
        raise FileNotFoundError(f'File not found: {path}')

    # Detect ISMRMRD containers
    def _is_ismrmrd(fh: h5py.File) -> bool:
        return any(k in fh for k in ('ismrmrd_header', 'dataset'))

    def _from_ismrmrd(path):
        if ismrmrd is None:
            raise RuntimeError('ismrmrd package not available. Install python-ismrmrd to read ISMRMRD files.')
        dset = ismrmrd.Dataset(path, 'dataset', create_if_needed=False)
        try:
            # Read XML header to get basic dims
            xml = dset.read_xml_header()
            header = ismrmrd_xsd.CreateFromDocument(xml)
            ncoils = header.acquisitionSystemInformation.receiverChannels
        except Exception:
            # Fallback if header missing
            ncoils = None

        n = dset.number_of_acquisitions()
        if n == 0:
            raise ValueError('No acquisitions found in ISMRMRD file.')

        # Read all acquisitions and stack by slice/phase-encoding order
        acq0 = dset.read_acquisition(0)
        nSamp = acq0.number_of_samples
        nCh = acq0.active_channels
        data = np.empty((n, nCh, nSamp), dtype=np.complex64)
        ky = np.empty(n, dtype=np.int32)
        kz = np.empty(n, dtype=np.int32)
        sl = np.empty(n, dtype=np.int32)
        for i in range(n):
            acq = dset.read_acquisition(i)
            data[i] = acq.data.astype(np.complex64)
            md = acq.idx
            ky[i] = getattr(md, 'kspace_encode_step_1', 0)
            kz[i] = getattr(md, 'kspace_encode_step_2', 0)
            sl[i] = getattr(md, 'slice', 0)
        dset.close()

        # Infer grid sizes
        Ny = int(ky.max() + 1) if ky.max() >= 0 else n
        Nz = int(kz.max() + 1) if kz.max() >= 0 else 1
        Nslices = int(sl.max() + 1) if sl.max() >= 0 else 1
        Nx = nSamp

        # Assemble to (S, C, H=Ny, W=Nx)
        S = Nslices * Nz
        kspace = np.zeros((S, nCh, Ny, Nx), dtype=np.complex64)

        # Map (slice,kz) -> slice index
        slice_map = {}
        s_idx = 0
        for s in range(Nslices):
            for z in range(Nz):
                slice_map[(s, z)] = s_idx
                s_idx += 1

        # Place lines
        order = np.lexsort((ky, kz, sl))
        for i in order:
            s = int(sl[i]); z = int(kz[i]); y = int(ky[i])
            si = slice_map[(s, z)]
            if 0 <= y < Ny:
                kspace[si, :, y, :] = data[i]

        ks_ri = np.stack((kspace.real, kspace.imag), axis=-1).astype(np.float32)
        info = {
            'filename': path,
            'dims': (int(S), int(nCh), int(Ny), int(Nx), 2),
            'n_slices': int(S),
            'n_coils': int(nCh),
            'format': 'ismrmrd'
        }
        return ks_ri, info

    # First, peek with h5py to decide the path, then close before proceeding
    is_ismrmrd_file = False
    ks = None
    with h5py.File(path, 'r') as f:
        is_ismrmrd_file = _is_ismrmrd(f)
        if not is_ismrmrd_file:
            # fastMRI-like path
            if 'kspace' in f:
                ks = f['kspace'][:]
            else:
                # Try to find first dataset
                def _find_first_dataset(g):
                    for k, v in g.items():
                        if isinstance(v, h5py.Dataset):
                            return g[k][:]
                        if isinstance(v, h5py.Group):
                            r = _find_first_dataset(v)
                            if r is not None:
                                return r
                    return None
                ks = _find_first_dataset(f)
                if ks is None:
                    raise KeyError('No dataset found in HDF5 file.')

    if is_ismrmrd_file:
        return _from_ismrmrd(path)

    # Process fastMRI-like array
    ks = np.asarray(ks)
    if np.iscomplexobj(ks):
        ks_ri = np.stack((ks.real, ks.imag), axis=-1)
    else:
        if ks.ndim >= 1 and ks.shape[-1] == 2:
            ks_ri = ks
        else:
            # If shape is (N,2) it's likely (samples, RI) per readout; try reshape to (S=1,C=1,H=N,W=1,2)
            if ks.ndim == 2 and ks.shape[-1] == 2:
                ks_ri = ks.reshape(1, 1, ks.shape[0], 1, 2)
            else:
                ks_ri = np.stack((ks, np.zeros_like(ks)), axis=-1)

    # Normalize shape to (S, C, H, W, 2)
    if ks_ri.ndim == 4:  # (S, H, W, 2) -> single-coil
        S, H, W, _ = ks_ri.shape
        ks_ri = ks_ri.reshape(S, 1, H, W, 2)
    elif ks_ri.ndim == 5:
        pass  # assume already (S, C, H, W, 2)
    else:
        raise ValueError(f'Unsupported k-space shape: {ks_ri.shape}')

    ks_ri = ks_ri.astype(np.float32, copy=False)

    # Build minimal metadata
    S, Cc, H, W, _ = ks_ri.shape
    info = {
        'filename': path,
        'dims': (int(S), int(Cc), int(H), int(W), 2),
        'n_slices': int(S),
        'n_coils': int(Cc),
        'format': 'fastmri-like'
    }
    # Try to attach optional header if present
    try:
        with h5py.File(path, 'r') as f2:
            print()
            if 'ismrmrd_header' in f2:
                hdr = f2['ismrmrd_header'][()]
                info['ismrmrd_header'] = hdr.decode('utf-8') if isinstance(hdr, (bytes, bytearray)) else str(hdr)
    except Exception:
        pass

    return ks_ri, info

In [ ]:
# Load data
if not filename:
    raise RuntimeError('No .h5 files found under data_path. Update data_path or set filename manually.')

kspace_np, info = load_h5_kspace(filename)
print('File:', info['filename'])
print('Format:', info.get('format', 'unknown'))
print('k-space shape (S, C, H, W, 2):', kspace_np.shape)
print('n_slices:', info['n_slices'], 'n_coils:', info['n_coils'])

In [ ]:
# Convert to torch and visualize recon & k-space magnitude per slice (works with new loader)
import torch
import matplotlib.pyplot as plt

# Expect shape (S, C, H, W, 2) from load_h5_kspace
kspace = torch.from_numpy(kspace_np).contiguous()
assert kspace.ndim == 5 and kspace.shape[-1] == 2, f"Expected (S, C, H, W, 2), got {tuple(kspace.shape)}"

S, Cc, H, W, _ = kspace.shape


def ifft2c_torch(x_ri: torch.Tensor) -> torch.Tensor:
    """
    Centered 2D iFFT for RI-paired tensors.
    Input: (..., H, W, 2) with last dim [real, imag].
    Returns: same shape (..., H, W, 2).
    """
    xc = torch.view_as_complex(x_ri.to(torch.float32))  # (..., H, W) complex64
    imgc = torch.fft.ifftshift(
        torch.fft.ifft2(torch.fft.fftshift(xc, dim=(-2, -1)), norm="ortho"),
        dim=(-2, -1),
    )
    return torch.view_as_real(imgc)  # (..., H, W, 2)


def rss_from_ri(x_ri: torch.Tensor, coil_dim: int = 0) -> torch.Tensor:
    """Root-sum-of-squares coil combine for RI-paired tensors with a coil dimension.
    Input: (C, H, W, 2)  -> Output: (H, W)
    """
    mag2 = x_ri[..., 0] ** 2 + x_ri[..., 1] ** 2  # (C, H, W)
    return torch.sqrt(mag2.sum(dim=coil_dim) + 1e-12)


with torch.no_grad():
    for s in range(S):
        ks_slice = kspace[s]  # (C, H, W, 2)

        # Reconstruct complex image via centered iFFT, then coil-combine with RSS
        img_ri = ifft2c_torch(ks_slice)              # (C, H, W, 2)
        img_mag = rss_from_ri(img_ri, coil_dim=0)    # (H, W)

        # k-space magnitude for display (log), collapse coils by max
        ks_c = torch.view_as_complex(ks_slice)
        ks_mag_log = torch.log1p(ks_c.abs()).amax(dim=0)  # (H, W)

        fig, axs = plt.subplots(1, 2, figsize=(10, 5), dpi=160)
        axs[0].imshow(img_mag.cpu().numpy(), cmap='gray')
        axs[0].set_title(f'Reconstructed Image - Slice {s+1}')
        axs[0].axis('off')
        axs[1].imshow(ks_mag_log.cpu().numpy(), cmap='gray')
        axs[1].set_title(f'k-space Magnitude (log) - Slice {s+1}')
        axs[1].axis('off')
        plt.show()